# Gold — physical_itens_venda_caixa

**KPIs de negócio gerados neste notebook (planilha "Squad 3 —
Negócio/Técnica", Luiz Henrique, regras 6-10):**

| # | KPI | Granularidade |
|---|-----|---------------|
| 6 | Receita por produto por loja por mês | produto + loja + ano + mês |
| 7 | Top 10 produtos por loja por trimestre | loja + trimestre |
| 8 | Crescimento MoM perecíveis vs secos por loja | loja + categoria + ano + mês |
| 9 | Quantidade média de itens por transação por loja | loja + ano + mês |
| 10 | Enriquecimento: flag `venda_em_feriado` | item |

**Sobre o KPI 8:** implementado a partir da coluna `categoria_produto`
(gerada na Silver via `classificar_produto()`, governança v1.1).
A classificação é por inferência sobre o nome da categoria-base do
`codigo_barras_produto`, **sem validação contra fonte oficial** —
ver nota de limitação completa em `governanca/00_data_quality_rules`.
Recomenda-se revisão humana da classificação antes de tratar este
KPI como definitivo para decisões de negócio.

**Dicionário de dados Gold (KPI 9 — documentação):**
- `id_item_venda`: identificador único do item (PK original).
- `id_transacao`: identificador da transação (FK para vendas_caixa).
- `id_loja`: identificador da loja associada à transação.
- `codigo_barras_produto`: código de barras do produto vendido.
- `categoria_produto`: classificação `perecivel` / `seco` /
  `NAO_CLASSIFICADO` (ver nota acima).
- `quantidade`: quantidade vendida no item.
- `preco_unitario_registro`: preço unitário registrado no sistema.
- `valor_item_analitico`: valor usado para KPIs de receita (calculado
  quando o original é inconsistente, original caso contrário).
- `dt_venda`: data da venda (vinda de physical_vendas_caixa).
- `ano`, `mes`, `trimestre`: derivados de `dt_venda`.
- `venda_em_feriado`: flag booleana indicando se a venda ocorreu
  em um feriado nacional brasileiro.
- `receita_produto_loja_mes`: soma de `valor_item_analitico` por
  produto + loja + mês (KPI 6).
- `rank_produto_trimestre`: ranking do produto por receita dentro
  da loja no trimestre (KPI 7, apenas top 10).
- `receita_categoria_mes`: receita por categoria (perecível/seco)
  por loja por mês (KPI 8).
- `crescimento_mom_categoria_pct`: variação percentual MoM da
  receita por categoria (KPI 8).
- `media_itens_transacao`: média de itens por transação por loja
  por mês (KPI 9).

**Regras de filtragem aplicadas nesta camada:**
- `flag_fk_invalido = true`: excluídos (sem loja/data associada).
- `flag_quantidade_invalido = true`: excluídos (distorce receita).
- `flag_preco_invalido = true`: excluídos (distorce receita).
- `flag_codigo_barras_ausente = true`: mantidos nos KPIs de receita
  agregada, excluídos apenas do KPI 7 (top 10 produtos), onde o
  código de barras é a chave de identidade do produto.
- `categoria_produto = "NAO_CLASSIFICADO"`: excluídos apenas do
  KPI 8 (não é possível atribuir a perecível ou seco).

**O que este notebook faz:**
- Lê da Silver (`squad3/silver/physical_itens_venda_caixa`).
- Aplica a flag `venda_em_feriado` (KPI 10) via lista de feriados
  nacionais hardcoded (2023-2026).
- Calcula KPIs 6, 7, 8, 9.
- Grava no SQL Server (`squad3.gold_physical_itens_venda_caixa`).
- Grava em Delta (`squad3/gold/physical_itens_venda_caixa`) para
  consumo pelos notebooks de Analysis.

In [0]:
%run "../utils/00_utils"

In [0]:
adls_options = get_adls_options()

# Modo de escrita da Gold:
#   "overwrite" -> reprocessamento completo (situação atual — dados
#                  residuais de execução anterior com erro de cota)
#   "append"    -> carga incremental (novos dados chegando no raw)
# Altere para "append" em execuções normais de atualização de dados.
# IMPORTANTE: a Gold no SQL Server usa sempre mode="overwrite" via
# write_sql_table_typed (substitui a tabela inteira a cada execução),
# independente deste modo — este controla apenas a escrita em Delta.
GOLD_WRITE_MODE = "overwrite"

In [0]:
from pyspark.sql.functions import (
    sum as spark_sum, count, countDistinct,
    avg, round as spark_round, year, month,
    quarter, when, col, lit, rank, lag, to_date, expr, first
)
from pyspark.sql.window import Window

## Leitura da Silver — physical_itens_venda_caixa

Aplicamos os filtros de qualidade decididos na Gold (ver regras
de filtragem no cabeçalho).

In [0]:
df_itens_silver = (
    read_delta(SILVER_ITENS_VENDA_CAIXA_PATH, adls_options)
    .filter(col("flag_fk_invalido") == False)
    .filter(col("flag_quantidade_invalido") == False)
    .filter(col("flag_preco_invalido") == False)
)

print(f"Itens válidos (Silver): {df_itens_silver.count():,}")

# Buscar id_loja e dt_venda da Bronze de vendas_caixa
# (id_loja na Silver de itens pode estar nulo — fonte confiável é vendas_caixa)
df_vendas_dim = (
    read_delta(BRONZE_VENDAS_CAIXA_PATH, adls_options)
    .select(
        col("id_transacao"),
        expr("TRY_CAST(id_loja AS BIGINT)").alias("id_loja"),
        col("dt_venda"),
    )
    .dropDuplicates(["id_transacao"])
)

print(f"Transações disponíveis: {df_vendas_dim.count():,}")

df_itens = (
    df_itens_silver
    .drop("id_loja", "dt_venda", "tipo_pagamento")
    .join(df_vendas_dim, on="id_transacao", how="inner")
    .withColumn("dt_venda_date", to_date(col("dt_venda")))
    .withColumn("ano",      year(col("dt_venda_date")))
    .withColumn("mes",      month(col("dt_venda_date")))
    .withColumn("trimestre", quarter(col("dt_venda_date")))
)

print(f"Itens após JOIN com vendas_caixa: {df_itens.count():,}")
display(df_itens.limit(5))

####Integridade referencial: id_loja órfão em vendas/itens

O join anterior usa BRONZE_VENDAS_CAIXA_PATH direto da Bronze (dado cru, sem validação) com inner join — um id_loja que não existe em physical_lojas passava silenciosamente para as agregações por loja, sem gerar nenhum alerta. Isso corrige exatamente o ponto do plano: *"todo id_loja usado em vendas/itens precisa existir em physical_lojas; ID órfão deve gerar flag visível, não sumir silenciosamente."*

A checagem usa a Silver de physical_lojas (fonte já tratada/validada) como lista de referência.



In [0]:
df_ids_lojas_validas = (
    read_delta(SILVER_LOJAS_PATH, adls_options)
    .select("id_loja")
    .distinct()
)

df_itens = (
    df_itens
    .join(
        df_ids_lojas_validas.withColumn("_loja_existe", lit(True)),
        on="id_loja",
        how="left",
    )
    .withColumn("flag_id_loja_orfao", col("_loja_existe").isNull())
    .drop("_loja_existe")
)

qtd_orfaos = df_itens.filter(col("flag_id_loja_orfao")).count()
total_itens_pre_filtro = df_itens.count()

if qtd_orfaos > 0:
    print(
        f"[ALERTA] {qtd_orfaos:,} de {total_itens_pre_filtro:,} item(ns) "
        f"referenciam um id_loja que NAO existe em physical_lojas. "
        f"Esses itens serao excluidos dos KPIs de loja (nao descartados "
        f"silenciosamente -- ficam registrados na metrica de DQ abaixo)."
    )
    display(
        df_itens.filter(col("flag_id_loja_orfao"))
        .select("id_loja", "id_transacao", "id_item_venda")
        .distinct()
        .limit(20)
    )
else:
    print("[OK] Nenhum id_loja orfao encontrado em vendas/itens.")

registrar_metrica_dq(
    tabela="physical_itens_venda_caixa",
    regra="11_id_loja_orfao_referencial",
    qtd_registros_afetados=qtd_orfaos,
    qtd_registros_total=total_itens_pre_filtro,
    adls_options=adls_options,
)

# Frente C: exclui os orfaos dos KPIs (nao existe loja pra atribuir a
# receita) mas o alerta acima e a metrica de DQ garantem visibilidade,
# em vez de sumir silenciosamente como no inner join anterior.
df_itens = df_itens.filter(~col("flag_id_loja_orfao"))

## Verificação de pré-condição

In [0]:
verificar_destino_limpo(
    GOLD_ITENS_VENDA_CAIXA_PATH,
    adls_options,
    permitir_existente=(GOLD_WRITE_MODE == "overwrite"),
)

## KPI 6 — Receita por produto por loja por mês

In [0]:
df_kpi_receita_produto = (
    df_itens
    .filter(col("flag_codigo_barras_ausente") == False)
    .groupBy(
        # Instrução do cliente: agrupa pelo código NORMALIZADO, não
        # pelo texto cru, para não diluir a posição de um produto por
        # variação de escrita (acento/espaço/hífen).
        "codigo_produto_normalizado", "id_loja", "ano", "mes", "trimestre"
    )
    .agg(
        spark_round(spark_sum("valor_item_analitico"), 2).alias("receita_produto_loja_mes"),
        spark_sum("quantidade").alias("qtd_vendida"),
        countDistinct("id_transacao").alias("qtd_transacoes"),
    )
)

print(f"KPI 6 — Receita por produto/loja/mês: {df_kpi_receita_produto.count()} linha(s).")
display(df_kpi_receita_produto.orderBy(col("receita_produto_loja_mes").desc()).limit(10))


## KPI 7 — Top 10 produtos por loja por trimestre

Usa função de janela `RANK()` para rankear os produtos dentro de
cada loja/trimestre por receita total. Mantém apenas o top 10.
Produtos sem código de barras são excluídos (não identificáveis).

In [0]:
df_receita_trimestre = (
    df_itens
    .filter(col("flag_codigo_barras_ausente") == False)
    .groupBy("codigo_produto_normalizado", "id_loja", "ano", "trimestre")
    .agg(
        spark_round(spark_sum("valor_item_analitico"), 2).alias("receita_trimestre"),
        spark_sum("quantidade").alias("qtd_vendida_trimestre"),
    )
)

janela_rank = Window.partitionBy("id_loja", "ano", "trimestre").orderBy(
    col("receita_trimestre").desc()
)

df_kpi_top10 = (
    df_receita_trimestre
    .withColumn("rank_produto_trimestre", rank().over(janela_rank))
    .filter(col("rank_produto_trimestre") <= 10)
)

print(f"KPI 7 — Top 10 produtos/trimestre: {df_kpi_top10.count()} linha(s).")
display(df_kpi_top10.orderBy("id_loja", "ano", "trimestre", "rank_produto_trimestre").limit(20))


## KPI 8 — Crescimento MoM perecíveis vs secos por loja

Agrega a receita por loja + categoria (`perecivel`/`seco`) + mês, e
usa `LAG()` para comparar com o mês anterior dentro da mesma
loja+categoria. Itens com `categoria_produto = "NAO_CLASSIFICADO"`
são excluídos deste KPI especificamente (não é possível atribuí-los
a perecível ou seco).

In [0]:
df_receita_categoria_mes = (
    df_itens
    .filter(col("categoria_produto") != "NAO_CLASSIFICADO")
    .groupBy("id_loja", "categoria_produto", "ano", "mes")
    .agg(
        spark_round(spark_sum("valor_item_analitico"), 2).alias("receita_categoria_mes"),
        countDistinct("id_transacao").alias("qtd_transacoes_categoria"),
    )
)

janela_mom_categoria = Window.partitionBy("id_loja", "categoria_produto").orderBy("ano", "mes")

df_kpi_categoria_mom = (
    df_receita_categoria_mes
    .withColumn(
        "receita_mes_anterior_categoria",
        lag("receita_categoria_mes", 1).over(janela_mom_categoria)
    )
    .withColumn(
        "crescimento_mom_categoria_pct",
        when(
            col("receita_mes_anterior_categoria").isNotNull()
            & (col("receita_mes_anterior_categoria") > 0),
            spark_round(
                (col("receita_categoria_mes") - col("receita_mes_anterior_categoria"))
                / col("receita_mes_anterior_categoria") * 100,
                2
            )
        ).otherwise(lit(None))
    )
)

print(f"KPI 8 — Perecível vs seco MoM: {df_kpi_categoria_mom.count()} linha(s).")
display(df_kpi_categoria_mom.orderBy("id_loja", "categoria_produto", "ano", "mes").limit(15))


## KPI 9 — Quantidade média de itens por transação por loja por mês

Calcula quantos itens (linhas de item_venda) existem em média por
transação, agrupado por loja e mês. Indica o "tamanho médio do
carrinho" de cada loja.

In [0]:
df_itens_por_transacao = (
    df_itens
    .groupBy("id_transacao", "id_loja", "ano", "mes")
    .agg(count("id_item_venda").alias("qtd_itens_transacao"))
)

df_kpi_media_itens = (
    df_itens_por_transacao
    .groupBy("id_loja", "ano", "mes")
    .agg(
        spark_round(avg("qtd_itens_transacao"), 2).alias("media_itens_transacao"),
        countDistinct("id_transacao").alias("qtd_transacoes"),
    )
)

print(f"KPI 9 — Média de itens/transação: {df_kpi_media_itens.count()} linha(s).")
display(df_kpi_media_itens.orderBy("id_loja", "ano", "mes").limit(10))


## Montagem da tabela Gold final

Consolida os KPIs calculados. A granularidade base é produto +
loja + ano + mês (KPI 6), com os demais KPIs desnormalizados via
JOIN.

**Nota sobre o KPI 8:** sua granularidade nativa é
loja + categoria + ano + mês (sem produto individual). Para evitar
duplicar `receita_categoria_mes` em cada produto da mesma
categoria/loja/mês, o JOIN traz primeiro `categoria_produto` (por
produto, vindo da Silver) e depois associa os agregados do KPI 8
por loja+categoria+ano+mês — cada linha do produto carrega o
contexto do KPI 8 da sua categoria, sem inflar a soma agregada.

In [0]:
df_produto_categoria = (
    df_itens
    .select("codigo_produto_normalizado", "categoria_produto")
    .distinct()
)

# Rótulo de exibição: variante de texto mais frequente do código cru
# para cada código normalizado — assim o gestor vê um nome legível,
# não o slug normalizado (sem espaço/acento), mas o agrupamento por
# baixo dos panos continua correto.
janela_exibicao = Window.partitionBy("codigo_produto_normalizado").orderBy(
    col("_qtd").desc()
)
df_produto_exibicao = (
    df_itens
    .groupBy("codigo_produto_normalizado", "codigo_barras_produto")
    .agg(count("*").alias("_qtd"))
    .withColumn("_rn", rank().over(janela_exibicao))
    .filter(col("_rn") == 1)
    .select(
        "codigo_produto_normalizado",
        col("codigo_barras_produto").alias("codigo_produto_exibicao"),
    )
)

df_gold = (
    df_kpi_receita_produto
    .join(
        df_kpi_top10.select(
            "codigo_produto_normalizado", "id_loja", "ano", "trimestre",
            "rank_produto_trimestre", "receita_trimestre",
            "qtd_vendida_trimestre"
        ),
        on=["codigo_produto_normalizado", "id_loja", "ano", "trimestre"],
        how="left",
    )
    .join(
        df_kpi_media_itens.select(
            "id_loja", "ano", "mes",
            "media_itens_transacao"
        ),
        on=["id_loja", "ano", "mes"],
        how="left",
    )
    .join(
        df_produto_categoria,
        on="codigo_produto_normalizado",
        how="left",
    )
    .join(
        df_produto_exibicao,
        on="codigo_produto_normalizado",
        how="left",
    )
    .join(
        df_kpi_categoria_mom.select(
            "id_loja", "categoria_produto", "ano", "mes",
            "receita_categoria_mes", "receita_mes_anterior_categoria",
            "crescimento_mom_categoria_pct",
        ),
        on=["id_loja", "categoria_produto", "ano", "mes"],
        how="left",
    )
    .select(
        "codigo_produto_normalizado",
        "codigo_produto_exibicao",
        "categoria_produto",
        "id_loja",
        "ano", "mes", "trimestre",
        "receita_produto_loja_mes",
        "qtd_vendida",
        "qtd_transacoes",
        "rank_produto_trimestre",
        "receita_trimestre",
        "qtd_vendida_trimestre",
        "receita_categoria_mes",
        "receita_mes_anterior_categoria",
        "crescimento_mom_categoria_pct",
        "media_itens_transacao",
        # Colunas de feriado NAO entram aqui -- essa tabela tem
        # granularidade produto+loja+mes, e o KPI 10 tem granularidade
        # loja+DIA. Ver notebook proprio 'gold_physical_feriado_dia'.
    )
)

print(f"Gold final: {df_gold.count()} linha(s).")
display(df_gold.limit(10))

## Escrita no Delta (Gold) e no SQL Server

In [0]:
(
(
    df_gold.write
    .format("delta")
    .options(**adls_options)
    .option("overwriteSchema", "true")
    .mode(GOLD_WRITE_MODE)
    .save(GOLD_ITENS_VENDA_CAIXA_PATH)
)
)

print(f"[OK] Delta Gold gravado em '{GOLD_ITENS_VENDA_CAIXA_PATH}'.")

In [0]:
write_sql_table_typed(df_gold, SQL_TABLE_GOLD_ITENS_VENDA_CAIXA)